# Load and select football data

Use all available leagues for **2022/23, 2023/24 and 2024/25**.
Matches, team statistics and pregame context stay in separate tables.
[Loading guide](../docs/analytics/season_loading.md) · [Table reference](../docs/analytics/season_inspection.md)

In [1]:
from pathlib import Path
from IPython.display import display
from xdiyo_analytics.data import inspect_season, load_seasons, list_stat_bundles, select_stats

project_root = Path('C:/Users/luisi/Documents/Programming/Python/xDiyo')
data_root = project_root / 'data/xDiyo_data'
seasons = ['22_23', '23_24', '24_25']
record_dir = project_root / 'experiment/initial_population/selections'

In [2]:
catalogue = inspect_season(data_root, 'Premier_League_24_25')
catalogue[['rows', 'column_count', 'description']]

,rows,column_count,description
table,,,
availability,3145,15,"Reported player availability/absence context, ..."
coverage,6080,8,Collection status and latest attempt status fo...
goal_actions,4190,21,"Actions within goal-linked sequences: actors, ..."
goal_sequences,1115,17,"Goal-linked sequences: scorer/team, classifica..."
heatmap_points,573983,11,"Team spatial points with coordinates, weights,..."
incidents,7647,14,"Match incidents: type/class, player, side and ..."
lineup_players,15188,16,"Player lineup entries: identity, playing posit..."
lineup_teams,760,10,"Team lineup context: formation, roster status,..."
matches,380,35,"Match identity, teams, competition, season, sc..."


In [3]:
print('Statistics columns:', catalogue.loc['statistics', 'columns'])
print('Pregame columns and types:', catalogue.loc['pregame', 'column_summary'])

Statistics columns: ['event_id', 'source_key', 'raw_hash', 'observed_at', 'period', 'group_name', 'key', 'name', 'side', 'display', 'value_type', 'statistics_type', 'source_item_json', 'team_id', 'group_order', 'stat_order', 'value', 'total']
Pregame columns and types: event_id: int64, source_key: string, raw_hash: string, observed_at: double, side: string, team_id: int64, position: double, value_json: string


## Load the selected seasons

`leagues=None` selects every available league. Each row gains `source_league` and `source_season`.
Choose `shots` independently when needed.
`pregame.position` is the provider's match-specific team position, not a complete standings table.

In [4]:
experiment = load_seasons(
    data_root, seasons, leagues=None,
    tables=['matches', 'statistics', 'pregame'], record_dir=record_dir,
)
{name: frame.shape for name, frame in experiment.tables.items()}

{'matches': (13976, 37), 'statistics': (3303574, 20), 'pregame': (27072, 10)}

In [5]:
print(f"{len(experiment.provenance['sources'])} publications across "
      f"{len(experiment.provenance['leagues'])} leagues")
match_counts = (
    experiment.matches.groupby(['source_league', 'source_season']).size()
    .unstack('source_season').reindex(columns=seasons)
)
match_counts

39 publications across 13 leagues


source_season,22_23,23_24,24_25
source_league,,,
Bundesliga,306,306,306
Bundesliga_2,306,306,306
Championship,552,552,552
Eredivisie,306,306,306
La_Liga,380,380,380
La_Liga_2,462,462,462
Ligue_1,380,306,306
Ligue_2,379,380,306
Premier_League,380,380,380


In [6]:
origin = ['source_league', 'source_season']
display(experiment.matches[origin + ['event_id', 'home_id', 'away_id']].head(3))
display(experiment.statistics[origin + ['event_id', 'period', 'group_name', 'key', 'side', 'value']].head(4))
display(experiment['pregame'][origin + ['event_id', 'side', 'position']].head(3))

,source_league,source_season,event_id,home_id,away_id
0,Bundesliga,22_23,10388221,2556,2569
1,Bundesliga,22_23,10388222,2542,2530
2,Bundesliga,22_23,10388223,2524,2674


,source_league,source_season,event_id,period,group_name,key,side,value
0,Bundesliga,22_23,10388221,ALL,Match overview,ballPossession,home,47.0
1,Bundesliga,22_23,10388221,ALL,Match overview,ballPossession,away,53.0
2,Bundesliga,22_23,10388221,ALL,Match overview,expectedGoals,home,1.17
3,Bundesliga,22_23,10388221,ALL,Match overview,expectedGoals,away,0.82


,source_league,source_season,event_id,side,position
0,Bundesliga,22_23,10388221,home,8.0
1,Bundesliga,22_23,10388221,away,16.0
2,Bundesliga,22_23,10388222,home,17.0


Selections are saved in `experiment/initial_population/selections`, outside the source data.
Reusing this directory pins each saved publication version. Discovery still includes newly available
matching publications; `experiment.provenance['sources']` lists the population loaded by this call.
An absent league-season combination stays absent. Counts describe collected matches and do not
establish provider-wide fixture completeness.

## Select statistic bundles

Combine bundles with exact `(period, group_name, key)` statistics. `totals` keeps provider `ALL`
rows; half bundles filter the same category's available observations. Defense means observed
defensive metrics. Standings remain in `pregame`. See the [selection guide](../docs/analytics/stat_selection.md).


In [7]:
list_stat_bundles()

{'all_all': 'all statistics; period=all available periods',
 'all_totals': 'all statistics; period=ALL',
 'all_first_half': 'all statistics; period=1ST',
 'all_second_half': 'all statistics; period=2ND',
 'attack_all': 'attack statistics; period=all available periods',
 'attack_totals': 'attack statistics; period=ALL',
 'attack_first_half': 'attack statistics; period=1ST',
 'attack_second_half': 'attack statistics; period=2ND',
 'defense_all': 'defense statistics; period=all available periods',
 'defense_totals': 'defense statistics; period=ALL',
 'defense_first_half': 'defense statistics; period=1ST',
 'defense_second_half': 'defense statistics; period=2ND',
 'all_stats': 'all statistics; period=all available periods',
 'all_totals_only': 'all statistics; period=ALL',
 'all_first_period_only': 'all statistics; period=1ST',
 'all_second_period_only': 'all statistics; period=2ND',
 'standings': 'Pregame team positions, kept in a separate table.'}

In [8]:
selected = select_stats(
    experiment,
    bundles=['attack_totals', 'defense_first_half', 'standings'],
    stats=[('ALL', 'Match overview', 'ballPossession')],
)
print({name: frame.shape for name, frame in selected.tables.items()})
display(selected.statistics[origin + ['event_id', 'period', 'group_name', 'key', 'side', 'value']].head(4))
display(selected.pregame[origin + ['event_id', 'side', 'position']].head(3))

{'matches': (13976, 37), 'statistics': (746880, 20), 'pregame': (27072, 7)}


,source_league,source_season,event_id,period,group_name,key,side,value
0,Bundesliga,22_23,10388221,ALL,Match overview,ballPossession,home,47.0
1,Bundesliga,22_23,10388221,ALL,Match overview,ballPossession,away,53.0
2,Bundesliga,22_23,10388221,ALL,Match overview,expectedGoals,home,1.17
3,Bundesliga,22_23,10388221,ALL,Match overview,expectedGoals,away,0.82


,source_league,source_season,event_id,side,position
0,Bundesliga,22_23,10388221,home,8.0
1,Bundesliga,22_23,10388221,away,16.0
2,Bundesliga,22_23,10388222,home,17.0
